# Machine Learning Final Lab – Complete Scikit-learn Practical Notebook

This notebook consolidates the topics discussed:
- Regression and classification
- Imports and model implementations
- Model working principles and important hyperparameters
- Dataset loading, preprocessing, splitting and scaling
- Regression and classification evaluation
- Confusion-matrix heatmaps
- Hyperparameter tuning with `GridSearchCV` (especially SVM)
- Parameter experiments and viva revision

**Before running:** Put your dataset CSV in the notebook's working directory and change `DATA_PATH` and `TARGET_COLUMN` in the setup cell. For the self-contained demonstrations, the notebook also includes built-in Scikit-learn datasets, so you can run those without an external CSV.

> Important: A hyperparameter is a setting chosen before/during model fitting (for example, `max_depth`). A model parameter is learned from data (for example, a Linear Regression coefficient).

## 1. Install and import libraries

Run the installation command only if a required package is missing.

In [ ]:
# If needed, install the libraries (run once):
# %pip install numpy pandas matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset splitting and cross-validation / hyperparameter search
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

# Preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Evaluation metrics
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

RANDOM_STATE = 42
print("Libraries imported successfully.")

## 2. Regression vs classification

| Regression | Classification |
|---|---|
| Predicts a continuous number | Predicts a class/category |
| Examples: price, marks, salary | Examples: spam/not spam, pass/fail |
| Metrics: MAE, MSE, RMSE, R² | Metrics: accuracy, precision, recall, F1 |

**Decision Trees have both versions:**
- `DecisionTreeRegressor`: predicts a numerical value.
- `DecisionTreeClassifier`: predicts a category.

The algorithm family is similar, but the prediction and splitting criteria differ.

## 3. Standard workflow for a supervised ML experiment

1. Load and inspect data.
2. Handle missing values and categorical columns.
3. Separate features `X` and target `y`.
4. Split into training and test sets.
5. Scale features when the algorithm needs it.
6. Create the model and choose hyperparameters.
7. Fit on training data.
8. Predict on test data.
9. Evaluate performance.

## 4. Load and inspect a CSV dataset

Edit the path and target column. This section is a template; it will work once those two values match your file.

In [ ]:
# EDIT THESE TWO VALUES FOR YOUR LAB DATASET
DATA_PATH = "data.csv"
TARGET_COLUMN = "target"

# Uncomment and run after placing your CSV in the notebook folder:
# df = pd.read_csv(DATA_PATH)
# display(df.head())
# print("Shape:", df.shape)       # (number of rows, number of columns)
# df.info()                       # column names, data types, non-null counts
# display(df.describe(include="all"))
# print("Missing values per column:\n", df.isnull().sum())
# print("Duplicate rows:", df.duplicated().sum())

### Common inspection commands

- `df.head()` – first five rows.
- `df.shape` – number of rows and columns.
- `df.info()` – data types and non-null counts.
- `df.describe()` – summary statistics for numerical columns.
- `df.isnull().sum()` – missing-value count per column.
- `df.duplicated().sum()` – number of duplicate rows.

Do not blindly delete missing rows. Choose a suitable imputation strategy based on the data and lab instructions.

## 5. Separate X and y; split the dataset

In [ ]:
# Example once df has been loaded:
# X = df.drop(columns=[TARGET_COLUMN])  # input features
# y = df[TARGET_COLUMN]                 # target/output

# For classification, stratify=y preserves approximate class proportions
# in both train and test sets. Do not use stratify for regression.
#
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=0.20,       # 20% test data; 80% training data
#     random_state=42,      # makes the random split reproducible
#     stratify=y            # classification only; remove for regression
# )

### Meaning of the common split parameters

- `test_size=0.2`: 20% goes to the test set and 80% to training.
- `train_size`: can specify the training fraction instead.
- `random_state=42`: fixes the random split so repeated runs use the same split.
- `stratify=y`: for classification, preserves approximately the same class proportions in each split.

**Avoid data leakage:** fit preprocessing steps (such as a scaler or imputer) using training data only. Then apply the learned transformation to test data.

## 6. Missing values, categorical data and feature scaling

In [ ]:
# Simple numerical and categorical preprocessing examples.
# Use only the columns/strategies appropriate for your dataset.

# Numerical missing-value example:
# df["Age"] = df["Age"].fillna(df["Age"].mean())

# Categorical missing-value example:
# df["City"] = df["City"].fillna(df["City"].mode()[0])

# Drop rows with any missing values (simple option; may discard useful data):
# df = df.dropna()

# LabelEncoder example for a target label (e.g., Pass/Fail):
# le = LabelEncoder()
# y_encoded = le.fit_transform(y)

# StandardScaler: transforms a feature approximately to mean 0 and standard
# deviation 1. Useful for SVM, KNN, Logistic Regression, Ridge and Lasso.
scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)  # learn from train + transform train
# X_test_scaled = scaler.transform(X_test)        # reuse train statistics on test

# MinMaxScaler: scales each feature to a chosen range, default [0, 1].
minmax = MinMaxScaler()
# X_train_minmax = minmax.fit_transform(X_train)
# X_test_minmax = minmax.transform(X_test)

### Which models usually need scaling?

| Model | Scaling? |
|---|---|
| Linear Regression | Often helpful when features have very different scales |
| Ridge / Lasso | Important |
| Logistic Regression | Important |
| KNN | Very important (distance-based) |
| SVC / SVR | Very important |
| Decision Tree | Usually unnecessary |
| Random Forest | Usually unnecessary |
| Tree-based Gradient Boosting | Usually unnecessary |
| Gaussian Naive Bayes | Depends on data; scaling is not generally required |

`fit_transform()` learns preprocessing values from the training data and transforms it. `transform()` applies those already learned values to new data.

# PART A – REGRESSION MODELS

## 7. Linear Regression

**Purpose:** predicts a continuous numerical value by fitting a straight-line relationship (or a hyperplane when there are multiple features). It chooses coefficients to minimize the sum of squared residuals.

For one feature, the prediction equation is:

\[
\hat y = b_0 + b_1x
\]

- \(b_0\): intercept – predicted output when \(x=0\).
- \(b_1\): slope – change in predicted output for a one-unit increase in \(x\).
- \(\hat y\): predicted value.

The ordinary least-squares fit minimizes the sum of squared differences between actual and predicted values.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression(
    fit_intercept=True,  # True: learn an intercept; False: force intercept to zero
    copy_X=True,         # copy X before fitting (mainly memory/data handling)
    positive=False       # True: constrain all learned coefficients to be non-negative
)

# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

# Learned values after fitting:
# model.coef_       -> learned feature coefficients
# model.intercept_  -> learned intercept

**Parameter effects**
- `fit_intercept=True`: estimates the intercept. Set `False` only when the data/model should pass through the origin.
- `copy_X=True`: controls whether a copy of input features is made; it does not normally change model quality.
- `positive=True`: forces coefficients to be non-negative; use only when justified by the problem.

Standard Scikit-learn `LinearRegression` does not use `learning_rate` or `epochs`.

## 8. Ridge Regression

Ridge is Linear Regression with an L2 penalty. It discourages very large coefficients and can reduce overfitting, especially when features are correlated.

Objective (conceptually):

\[
\text{Loss} = \text{MSE} + \alpha \sum_j \beta_j^2
\]

`alpha` controls regularization strength.

In [ ]:
from sklearn.linear_model import Ridge

model = Ridge(
    alpha=1.0,             # larger alpha -> stronger L2 penalty, smaller coefficients
    fit_intercept=True,   # estimate intercept
    solver="auto",        # let sklearn choose a suitable optimization solver
    random_state=42       # used by certain solvers; harmless for others
)
# model.fit(X_train_scaled, y_train)
# y_pred = model.predict(X_test_scaled)

**Alpha effect:** `alpha=0` is equivalent to ordinary least squares (use `LinearRegression` for that case); small alpha means weak regularization; large alpha means strong shrinkage. Extremely large alpha can underfit. Feature scaling is important for fair regularization across features.

## 9. Lasso Regression

Lasso uses an L1 penalty. It shrinks coefficients and can set some coefficients exactly to zero, so it can perform feature selection.

\[
\text{Loss} = \text{MSE} + \alpha \sum_j |\beta_j|
\]

In [ ]:
from sklearn.linear_model import Lasso

model = Lasso(
    alpha=1.0,          # larger alpha -> stronger L1 penalty; more coefficients may become zero
    fit_intercept=True,
    max_iter=1000,      # maximum optimization iterations
    tol=0.0001,         # optimization stopping tolerance; smaller is stricter
    selection="cyclic", # "cyclic" or "random" coordinate update order
    random_state=42     # relevant when selection="random"
)
# model.fit(X_train_scaled, y_train)
# y_pred = model.predict(X_test_scaled)

**Ridge vs Lasso:** Ridge (L2) generally keeps all features with smaller coefficients. Lasso (L1) can set some coefficients exactly to zero. Larger `alpha` strengthens either penalty and can cause underfitting.

## 10. Decision Tree Regressor

A regression tree repeatedly splits the feature space into regions using feature-value questions. Each leaf predicts a numerical value (commonly the average target value in that leaf).

In [ ]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(
    criterion="squared_error", # split quality: squared_error, friedman_mse,
                               # absolute_error, or poisson (Poisson needs suitable positive targets)
    splitter="best",           # "best" chooses best split; "random" chooses a random split
    max_depth=3,                # maximum tree depth; None allows growth until stopping rules apply
    min_samples_split=2,        # minimum samples required to split an internal node
    min_samples_leaf=1,        # minimum samples required in each leaf
    max_features=None,          # features considered at each split; None means all
    max_leaf_nodes=None,        # optional maximum number of leaf nodes
    random_state=42
)
# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

**Parameter effects**
- `max_depth` ↑: tree can become more complex; may overfit.
- `min_samples_split` ↑: fewer splits; simpler tree; may underfit if too high.
- `min_samples_leaf` ↑: larger leaves and smoother predictions.
- `max_leaf_nodes` ↓: limits complexity.
- `criterion`: changes how regression splits are evaluated.
- `splitter`: `"best"` chooses the best split; `"random"` considers random splits.

## 11. Random Forest Regressor

Random Forest trains many decision trees on bootstrap samples and averages their numerical predictions. Averaging usually reduces the variance of a single tree.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,       # number of trees; more trees often improve stability but take longer
    criterion="squared_error", # split criterion: squared_error, absolute_error,
                               # friedman_mse, or poisson (target constraints apply)
    max_depth=5,            # maximum depth of each tree; None allows unrestricted growth
    min_samples_split=2,    # minimum samples needed to split a node
    min_samples_leaf=1,     # minimum samples in a leaf
    max_features=1.0,       # fraction/number of features considered per split; 1.0 = all features
    bootstrap=True,         # True: train trees on bootstrap samples
    random_state=42,
    n_jobs=-1               # use all available CPU cores
)
# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

**Parameter effects:** more `n_estimators` generally stabilizes results but costs computation; greater `max_depth` increases tree complexity; larger `min_samples_split` / `min_samples_leaf` regularizes trees; `max_features` controls tree diversity; `bootstrap` controls sampling with replacement.

## 12. K-Nearest Neighbors Regressor

KNN finds the nearest training samples to a new input and averages their target values (or uses distance weights). It is distance-based, so feature scaling is usually essential.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

model = KNeighborsRegressor(
    n_neighbors=5,       # number of nearest training points used
    weights="uniform",   # "uniform": equal vote; "distance": closer points count more
    algorithm="auto",    # "auto", "ball_tree", "kd_tree", or "brute"
    leaf_size=30,        # tree-search speed/memory setting for BallTree/KDTree
    p=2,                 # Minkowski power: p=1 Manhattan; p=2 Euclidean
    metric="minkowski"   # distance metric
)
# model.fit(X_train_scaled, y_train)
# y_pred = model.predict(X_test_scaled)

**K effect:** K=1 is very sensitive to individual points and may overfit. Increasing K smooths predictions; an excessively large K may underfit. `weights="distance"` gives closer neighbours more influence. `p=1` is Manhattan distance; `p=2` is Euclidean distance.

## 13. Support Vector Regression (SVR)

SVR fits a function while allowing errors within an epsilon-wide tube around the prediction function. It can model linear or nonlinear relationships through kernels.

In [ ]:
from sklearn.svm import SVR

model = SVR(
    kernel="rbf",      # "linear", "poly", "rbf", or "sigmoid"
    C=1.0,             # penalty strength: higher C penalizes errors more strongly
    epsilon=0.1,       # width of the no-penalty error tube; larger epsilon ignores more small errors
    gamma="scale",     # RBF/poly/sigmoid influence: "scale", "auto", or numeric value
    degree=3           # polynomial degree; used only with kernel="poly"
)
# model.fit(X_train_scaled, y_train)
# y_pred = model.predict(X_test_scaled)

**Parameter effects:** higher `C` fits training data more strictly and may overfit; larger `epsilon` makes the model less sensitive to small errors; larger RBF `gamma` makes each sample's influence more localized and can create a more complex fit. Scaling is very important for SVR.

## 14. Gradient Boosting Regressor

Gradient Boosting builds small trees sequentially. Each new tree helps correct errors made by the current ensemble.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(
    loss="squared_error", # regression loss: squared_error, absolute_error, huber, quantile
    learning_rate=0.1,    # contribution of each tree; lower values usually need more trees
    n_estimators=100,     # number of boosting stages/trees
    subsample=1.0,        # fraction of training samples per tree; <1 adds randomness
    max_depth=3,          # depth of each individual regression tree
    random_state=42
)
# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

**Parameter effects:** lower `learning_rate` means each tree contributes less and usually requires more `n_estimators`; too many trees or deep trees can overfit; `subsample<1` adds stochasticity and can reduce variance.

# PART B – CLASSIFICATION MODELS

## 15. Logistic Regression

Despite its name, Logistic Regression is a classification algorithm. It estimates class probabilities using a logistic (sigmoid) function for binary classification, then predicts a class using a threshold (often 0.5).

For binary classification:

\[
P(y=1)=\frac{1}{1+e^{-z}}
\]

The default class decision is generally based on a probability threshold of 0.5.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    C=1.0,              # inverse regularization strength: smaller C = stronger regularization
    penalty="l2",       # regularization: "l1", "l2", "elasticnet", or None (solver restrictions apply)
    solver="lbfgs",     # optimization algorithm; solver determines supported penalties
    max_iter=100,       # maximum optimization iterations; increase if convergence warning appears
    class_weight=None,  # None or "balanced"; balanced adjusts weights for class imbalance
    tol=0.0001,         # convergence tolerance; smaller means stricter convergence
    random_state=42
)
# model.fit(X_train_scaled, y_train)
# y_pred = model.predict(X_test_scaled)
# probabilities = model.predict_proba(X_test_scaled)

**Important:** `C` is inverse regularization strength. Small C = stronger regularization and simpler model; large C = weaker regularization and more freedom to fit. `max_iter` is an optimization limit, not a number of training epochs in the neural-network sense. `penalty` and `solver` must be compatible.

## 16. Decision Tree Classifier

A classification tree asks feature-value questions and predicts the majority class in each leaf.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    criterion="gini",       # "gini", "entropy", or "log_loss" measure class impurity
    splitter="best",        # "best" or "random"
    max_depth=3,             # maximum tree depth
    min_samples_split=2,     # minimum samples required to split a node
    min_samples_leaf=1,      # minimum samples in a leaf
    max_features=None,       # features considered at each split
    class_weight=None,       # None or "balanced" for class imbalance
    random_state=42
)
# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

**Parameter effects:** greater `max_depth` allows more complex rules and may overfit; larger `min_samples_split` / `min_samples_leaf` makes the tree simpler; `criterion` chooses the impurity measure. Gini and entropy both measure class impurity; a pure node has impurity zero.

Gini impurity:

\[
Gini = 1-\sum_i p_i^2
\]

Entropy:

\[
Entropy=-\sum_i p_i\log_2(p_i)
\]

## 17. Random Forest Classifier

Random Forest trains many decision trees and combines their class predictions by majority vote (and can average class probabilities).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,       # number of trees
    criterion="gini",       # "gini", "entropy", or "log_loss"
    max_depth=5,            # maximum depth of each tree
    min_samples_split=2,    # minimum samples to split a node
    min_samples_leaf=1,     # minimum samples in a leaf
    max_features="sqrt",    # number/fraction of features considered at each split
    bootstrap=True,         # use bootstrap samples for each tree
    class_weight=None,      # None, "balanced", or "balanced_subsample"
    random_state=42,
    n_jobs=-1
)
# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

**Parameter effects:** more trees often improve stability but take longer; deeper trees can learn more complex patterns; larger minimum sample settings regularize; `max_features` changes tree diversity; class weights can help imbalanced classes. More trees do not guarantee higher test accuracy.

## 18. K-Nearest Neighbors Classifier

KNN classifies a new point using the majority class among its nearest training points. It is distance-based; scale features first.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(
    n_neighbors=5,       # number of nearest points voting
    weights="uniform",   # equal vote, or "distance" for closer points to count more
    algorithm="auto",    # "auto", "ball_tree", "kd_tree", or "brute"
    leaf_size=30,        # search-tree speed/memory setting
    p=2,                 # Minkowski distance: 1 Manhattan, 2 Euclidean
    metric="minkowski"
)
# model.fit(X_train_scaled, y_train)
# y_pred = model.predict(X_test_scaled)

**K effect:** low K can overfit/noise sensitivity; high K makes a smoother boundary but may underfit. `weights='distance'` gives nearby samples more influence.

## 19. Support Vector Classifier (SVC)

SVC finds a decision boundary (hyperplane) that separates classes while aiming for a large margin. Kernels allow nonlinear boundaries.

In [ ]:
from sklearn.svm import SVC

model = SVC(
    C=1.0,               # error penalty: larger C penalizes misclassification more strongly
    kernel="rbf",        # "linear", "poly", "rbf", or "sigmoid"
    degree=3,             # polynomial degree; used only for kernel="poly"
    gamma="scale",       # RBF/poly/sigmoid: "scale", "auto", or numeric value
    probability=False,   # True enables predict_proba; adds extra fitting cost
    class_weight=None,   # None or "balanced" for class imbalance
    random_state=42
)
# model.fit(X_train_scaled, y_train)
# y_pred = model.predict(X_test_scaled)

**SVC parameter effects**
- `C`: higher C penalizes training errors more strongly; can create a tighter fit and may overfit.
- `kernel`: `linear` makes a linear boundary; `rbf` can make nonlinear boundaries; `poly` uses polynomial relationships; `sigmoid` is another kernel option.
- `gamma`: for RBF/poly/sigmoid, larger gamma makes influence more localized and boundaries more complex; smaller gamma makes influence broader.
- `degree`: controls polynomial degree when `kernel='poly'`.
- `probability=True`: enables probability estimates through additional calibration work.

SVM is very sensitive to feature scales.

## 20. Gaussian Naive Bayes

Gaussian Naive Bayes applies Bayes' theorem and assumes each numerical feature follows a Gaussian (normal) distribution within each class. It also assumes conditional independence of features given the class.

Bayes' theorem:

\[
P(A|B)=\frac{P(B|A)P(A)}{P(B)}
\]

In [ ]:
from sklearn.naive_bayes import GaussianNB

model = GaussianNB(
    priors=None,          # optional prior class probabilities; None learns priors from data
    var_smoothing=1e-9    # adds a small amount to feature variances for numerical stability
)
# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

**Parameter effects:** custom `priors` change prior class probabilities; larger `var_smoothing` adds more smoothing to variance estimates. GaussianNB is intended for numerical features with roughly Gaussian class-conditional distributions.

## 21. Gradient Boosting Classifier

Gradient Boosting builds trees sequentially. Each new tree helps correct the errors of the existing ensemble.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(
    loss="log_loss",       # classification loss; log_loss is the standard probabilistic loss
    learning_rate=0.1,     # contribution of each tree; lower often requires more trees
    n_estimators=100,      # number of boosting stages
    subsample=1.0,         # fraction of training data used per stage; <1 adds randomness
    max_depth=3,           # depth of individual trees
    random_state=42
)
# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

**Parameter effects:** lower learning rate means slower learning and usually requires more trees; more/deeper trees increase complexity and can overfit; subsample below 1 introduces randomness.

# PART C – EVALUATION

## 22. Regression evaluation metrics

Use these metrics when the target is numerical.

In [ ]:
# After fitting a regression model:
# y_pred = model.predict(X_test)

# MAE: average absolute prediction error. Lower is better.
# mae = mean_absolute_error(y_test, y_pred)

# MSE: average squared error; large errors are penalized more.
# mse = mean_squared_error(y_test, y_pred)

# RMSE: square root of MSE; has the same unit as the target.
# rmse = np.sqrt(mse)

# R-squared: proportion of target variation explained relative to predicting
# the test-set mean. 1 is perfect; 0 matches the mean baseline; negative is worse.
# r2 = r2_score(y_test, y_pred)

# print("MAE:", mae)
# print("MSE:", mse)
# print("RMSE:", rmse)
# print("R2:", r2)

| Metric | Meaning | Preferred |
|---|---|---|
| MAE | Mean absolute error | Lower |
| MSE | Mean squared error | Lower |
| RMSE | Square root of MSE; target's unit | Lower |
| R² | Fit relative to mean baseline | Higher (1 is perfect) |

R² can be negative on test data if the model performs worse than simply predicting the test target mean.

## 23. Classification evaluation metrics

Use these metrics when the target is a class/category.

In [ ]:
# After fitting a classifier:
# y_pred = model.predict(X_test)

# Accuracy: fraction of all predictions that are correct.
# print("Accuracy:", accuracy_score(y_test, y_pred))

# Confusion matrix: rows = actual classes; columns = predicted classes.
# cm = confusion_matrix(y_test, y_pred)
# print(cm)

# Precision: among predicted positives, fraction that are truly positive.
# print("Precision:", precision_score(y_test, y_pred, zero_division=0))

# Recall: among actual positives, fraction correctly identified.
# print("Recall:", recall_score(y_test, y_pred, zero_division=0))

# F1: harmonic mean of precision and recall.
# print("F1:", f1_score(y_test, y_pred, zero_division=0))

# Full per-class metrics and support:
# print(classification_report(y_test, y_pred, zero_division=0))

For binary classification:
- TP: actual positive, predicted positive.
- TN: actual negative, predicted negative.
- FP: actual negative, predicted positive.
- FN: actual positive, predicted negative.

Accuracy is not always sufficient, especially when classes are imbalanced. Precision matters when false positives are costly; recall matters when false negatives are costly; F1 balances precision and recall.

## 24. Confusion matrix heatmap – understand every argument

In [ ]:
# Example confusion matrix:
cm = np.array([[50, 5],
               [10, 35]])

plt.figure(figsize=(5, 4))

sns.heatmap(
    cm,                 # 2D matrix containing confusion-matrix counts
    annot=True,         # True: write the numerical value inside every cell
    fmt="d",            # format values as integers ("d"); use ".2f" for 2 decimals
    cmap="Blues",       # colour map; alternatives: "Reds", "Greens", "viridis", "coolwarm"
    cbar=True           # show the colour scale bar (optional)
)

plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.title("Confusion Matrix")
plt.show()

**Line explained:** `sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")`
- `sns`: Seaborn alias (`import seaborn as sns`).
- `heatmap()`: draws matrix values using colour intensity.
- `cm`: the matrix to display.
- `annot=True`: displays numbers inside cells; `False` hides them.
- `fmt="d"`: displays integer counts; `".2f"` displays two decimal places.
- `cmap="Blues"`: chooses blue shades; other maps include `"Reds"`, `"Greens"`, `"viridis"`, `"coolwarm"`.
- A darker cell generally means a larger count, according to the selected colour map.

For binary labels ordered `[0, 1]`, the standard matrix layout is `[[TN, FP], [FN, TP]]`.

# PART D – HYPERPARAMETER TUNING

## 25. What is hyperparameter tuning?

Hyperparameters are settings selected before model fitting, such as SVM's `C`, `kernel`, and `gamma`. Hyperparameter tuning tests candidate settings and selects the combination with the best cross-validation score. The test set should be kept separate and used only for final evaluation.

## 26. GridSearchCV – general pattern

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

base_model = DecisionTreeClassifier(random_state=42)

# Dictionary: parameter names -> candidate values to test
param_grid = {
    "max_depth": [2, 3, 5, None],
    "min_samples_split": [2, 5, 10],
    "criterion": ["gini", "entropy"]
}

grid = GridSearchCV(
    estimator=base_model,  # model to tune
    param_grid=param_grid, # all candidate parameter values
    cv=5,                  # 5-fold cross-validation
    scoring="accuracy",    # metric used to compare combinations
    verbose=1,             # display progress messages
    n_jobs=-1              # use all available CPU cores
)

# grid.fit(X_train, y_train)
# print("Best parameters:", grid.best_params_)
# print("Best mean CV score:", grid.best_score_)
# best_model = grid.best_estimator_
# y_pred = best_model.predict(X_test)

**GridSearchCV arguments and results**
- `estimator`: the model to tune.
- `param_grid`: dictionary of parameter names and candidate values.
- `cv=5`: five-fold cross-validation; each combination is evaluated across five validation folds.
- `scoring`: metric used to select the best combination (e.g. `"accuracy"`, `"f1"`, `"recall"`, `"r2"`, `"neg_mean_squared_error"`).
- `verbose`: controls progress output.
- `n_jobs=-1`: uses all available CPU cores.
- `best_params_`: best parameter combination.
- `best_score_`: best mean cross-validation score.
- `best_estimator_`: fitted estimator with the best settings (by default refit on all training data).

If there are 3 values for C and 2 kernel values, there are 3 × 2 = 6 combinations. With `cv=5`, GridSearchCV performs 6 × 5 = 30 cross-validation fits (plus the final refit when `refit=True`, the default).

## 27. SVM hyperparameter tuning using GridSearchCV – exam-ready template

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Assume X contains numeric features and y contains class labels.
# Split first. For classification, stratify preserves class proportions.
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )

# Best practice: put scaling and SVC in a Pipeline.
# This ensures the scaler is fitted separately inside each CV training fold,
# preventing data leakage during GridSearchCV.
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC())
])

# Pipeline parameter names use: step_name + double underscore + parameter.
param_grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__kernel": ["linear", "rbf", "poly"],
    "svc__gamma": ["scale", "auto", 0.1, 1],
    # degree matters only when kernel="poly"; it is harmless to list here,
    # but a list-of-dictionaries can avoid testing irrelevant combinations.
}

grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    cv=5,                 # 5-fold cross-validation on training data
    scoring="accuracy",   # choose best combination by mean validation accuracy
    verbose=1,
    n_jobs=-1,
    refit=True            # refit best pipeline on all training data
)

# Fit/tune:
# grid.fit(X_train, y_train)

# Inspect search result:
# print("Best parameters:", grid.best_params_)
# print("Best CV accuracy:", grid.best_score_)

# grid.best_estimator_ is the best fitted pipeline (scaler + SVC).
# Predict on the untouched test set:
# y_pred = grid.best_estimator_.predict(X_test)
# print("Test accuracy:", accuracy_score(y_test, y_pred))
# print(classification_report(y_test, y_pred))

### SVC parameters to understand

| Parameter | Meaning | Typical effect |
|---|---|---|
| `C` | Penalty for training errors | Higher C penalizes errors more strongly; may overfit |
| `kernel` | Shape of decision boundary | `linear`, `rbf`, `poly`, `sigmoid` |
| `gamma` | Influence scale for RBF/poly/sigmoid | Higher gamma gives more localized influence and potentially more complex boundaries |
| `degree` | Polynomial degree | Used only with `kernel="poly"` |
| `probability` | Enables `predict_proba()` | Adds fitting/calibration cost |
| `class_weight` | Class importance | `"balanced"` adjusts weights for imbalanced classes |

For a basic `SVC()` grid without a pipeline, the parameter names are `"C"`, `"kernel"`, and `"gamma"`. In the Pipeline above they must be `"svc__C"`, `"svc__kernel"`, and `"svc__gamma"`.

## 28. Smaller SVM grid with only relevant parameter combinations

This avoids searching gamma for a linear kernel, where gamma is ignored, and tests polynomial degree only for the polynomial kernel.

In [ ]:
param_grid_svm = [
    {
        "svc__kernel": ["linear"],
        "svc__C": [0.1, 1, 10, 100]
    },
    {
        "svc__kernel": ["rbf"],
        "svc__C": [0.1, 1, 10, 100],
        "svc__gamma": ["scale", "auto", 0.1, 1]
    },
    {
        "svc__kernel": ["poly"],
        "svc__C": [0.1, 1, 10],
        "svc__gamma": ["scale", "auto"],
        "svc__degree": [2, 3, 4]
    }
]

# grid = GridSearchCV(
#     svm_pipeline, param_grid_svm, cv=5,
#     scoring="accuracy", n_jobs=-1, verbose=1
# )
# grid.fit(X_train, y_train)
# print(grid.best_params_)
# print(grid.best_score_)

## 29. Experiment: see how SVM parameters affect accuracy

Use this after defining `X_train`, `X_test`, `y_train`, `y_test`. This compares combinations using a pipeline so scaling is performed correctly.

In [ ]:
# Example manual experiment (not a substitute for cross-validation):
# for c_value in [0.1, 1, 10, 100]:
#     experiment = Pipeline([
#         ("scaler", StandardScaler()),
#         ("svc", SVC(C=c_value, kernel="rbf", gamma="scale"))
#     ])
#     experiment.fit(X_train, y_train)
#     pred = experiment.predict(X_test)
#     print("C =", c_value, "Test accuracy =", accuracy_score(y_test, pred))
#
# Do not choose hyperparameters solely because one test-set result is highest.
# Use cross-validation on training data for tuning, then evaluate once on test data.

# PART E – OPTIONAL RUNNABLE DEMONSTRATIONS

## 30. Runnable classification example using the built-in Iris dataset

This example does not need an external CSV. It demonstrates split → pipeline → SVM GridSearchCV → test evaluation → heatmap.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
X_demo = iris.data
y_demo = iris.target

X_train_demo, X_test_demo, y_train_demo, y_test_demo = train_test_split(
    X_demo, y_demo,
    test_size=0.2,
    random_state=42,
    stratify=y_demo
)

demo_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC())
])

demo_grid = {
    "svc__C": [0.1, 1, 10],
    "svc__kernel": ["linear", "rbf"],
    "svc__gamma": ["scale", "auto"]
}

demo_search = GridSearchCV(
    demo_pipeline,
    demo_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

demo_search.fit(X_train_demo, y_train_demo)

demo_pred = demo_search.predict(X_test_demo)

print("Best parameters:", demo_search.best_params_)
print("Best CV accuracy:", demo_search.best_score_)
print("Test accuracy:", accuracy_score(y_test_demo, demo_pred))
print(classification_report(y_test_demo, demo_pred, target_names=iris.target_names))

cm_demo = confusion_matrix(y_test_demo, demo_pred)
sns.heatmap(cm_demo, annot=True, fmt="d", cmap="Blues",
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Iris – SVM Confusion Matrix")
plt.show()

## 31. Runnable regression example using the built-in diabetes dataset

In [ ]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes(as_frame=True)
X_reg_demo = diabetes.data
y_reg_demo = diabetes.target

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg_demo, y_reg_demo,
    test_size=0.2,
    random_state=42
)

reg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

reg_grid = {
    "model__alpha": [0.01, 0.1, 1, 10, 100]
}

reg_search = GridSearchCV(
    reg_pipeline,
    reg_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

reg_search.fit(Xr_train, yr_train)
reg_pred = reg_search.predict(Xr_test)

print("Best parameters:", reg_search.best_params_)
print("Best CV R2:", reg_search.best_score_)
print("Test MAE:", mean_absolute_error(yr_test, reg_pred))
print("Test MSE:", mean_squared_error(yr_test, reg_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(yr_test, reg_pred)))
print("Test R2:", r2_score(yr_test, reg_pred))

# PART F – QUICK REVISION

## 32. Model and main hyperparameter summary

| Model | Task | Main hyperparameters to remember |
|---|---|---|
| Linear Regression | Regression | `fit_intercept`, `positive` |
| Ridge | Regression | `alpha` |
| Lasso | Regression | `alpha`, `max_iter`, `tol` |
| Decision Tree Regressor | Regression | `max_depth`, `min_samples_split`, `min_samples_leaf`, `criterion` |
| Random Forest Regressor | Regression | `n_estimators`, `max_depth`, `max_features` |
| KNN Regressor | Regression | `n_neighbors`, `weights`, `metric`, `p` |
| SVR | Regression | `C`, `kernel`, `epsilon`, `gamma` |
| Gradient Boosting Regressor | Regression | `n_estimators`, `learning_rate`, `max_depth`, `subsample` |
| Logistic Regression | Classification | `C`, `penalty`, `solver`, `max_iter`, `class_weight` |
| Decision Tree Classifier | Classification | `criterion`, `max_depth`, `min_samples_split`, `min_samples_leaf` |
| Random Forest Classifier | Classification | `n_estimators`, `max_depth`, `max_features`, `class_weight` |
| KNN Classifier | Classification | `n_neighbors`, `weights`, `metric`, `p` |
| SVC | Classification | `C`, `kernel`, `gamma`, `degree` |
| Gaussian Naive Bayes | Classification | `priors`, `var_smoothing` |
| Gradient Boosting Classifier | Classification | `n_estimators`, `learning_rate`, `max_depth`, `subsample` |

## 33. The three essential Scikit-learn steps

```python
model = ModelName(parameters)   # create model and choose hyperparameters
model.fit(X_train, y_train)     # train/fit model using training data
y_pred = model.predict(X_test)  # predict outputs for unseen test features
```

## 34. Common viva questions and answers

1. **Regression vs classification?** Regression predicts continuous numbers; classification predicts categories.
2. **`fit()` vs `predict()`?** `fit()` learns from training data; `predict()` predicts outputs for new inputs.
3. **Why train-test split?** To evaluate on unseen data.
4. **What is `random_state`?** A seed for reproducible random operations.
5. **What is overfitting?** Very good training performance but poor performance on unseen data.
6. **What is underfitting?** Model is too simple and performs poorly on both training and test data.
7. **Why StandardScaler?** It standardizes features to approximately zero mean and unit variance.
8. **Random Forest vs Decision Tree?** Random Forest combines many trees; this often reduces variance.
9. **Ridge vs Lasso?** Ridge uses L2; Lasso uses L1 and can zero some coefficients.
10. **What happens when K increases in KNN?** Predictions/boundaries become smoother; too-large K may underfit.
11. **MAE vs MSE?** MAE uses absolute errors; MSE squares errors and penalizes large errors more.
12. **Precision vs recall?** Precision measures correctness among predicted positives; recall measures identified positives among actual positives.
13. **Random Forest vs Gradient Boosting?** Random Forest trees are built independently; Gradient Boosting builds trees sequentially to correct previous errors.
14. **What is GridSearchCV?** Exhaustively tests specified hyperparameter combinations using cross-validation.
15. **What does `cv=5` mean?** Five-fold cross-validation.
16. **What is `best_params_`?** The best parameter combination found.
17. **What is `best_score_`?** The best mean cross-validation score.
18. **What is `best_estimator_`?** The estimator fitted with the selected best parameters.
19. **Why scale SVM/KNN data?** Both depend on distances or feature magnitudes.
20. **Why use a Pipeline in GridSearchCV?** It ensures preprocessing is fitted inside each cross-validation training fold, preventing leakage.

## 35. Final practical checklist

- [ ] Identify whether the task is regression or classification.
- [ ] Identify the target column and feature columns.
- [ ] Inspect missing values and data types.
- [ ] Encode categorical data if needed.
- [ ] Split data before fitting preprocessing.
- [ ] Use `stratify=y` for classification when appropriate.
- [ ] Scale features for SVM, KNN and regularized linear models.
- [ ] Fit the model on training data only.
- [ ] Predict on test data.
- [ ] Use suitable evaluation metrics.
- [ ] Tune hyperparameters using cross-validation on training data.
- [ ] Keep the test set for final evaluation.

**Exam reminder:** Parameter changes do not guarantee improved results. Compare cross-validation performance, check for underfitting/overfitting, and evaluate the selected model on the untouched test set.